# EXP-002 — LR baseline

## 01 — Thông tin
Hồ sơ theo kế hoạch chung EXP-001–020. Phân tích kết quả chạy lại đã lưu; ngày đồng bộ 19/11/2026 không phải ngày huấn luyện.



In [1]:
from pathlib import Path
import json
import hashlib
import yaml
import pandas as pd
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/wesad/models.py").is_file())
CONFIG_PATH = ROOT / 'experiments/01_baselines/EXP-002_LR/config.yaml'
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
RESULTS_PATH = ROOT / 'experiments/01_baselines/EXP-002_LR/results.csv'
FOLDER = CONFIG_PATH.parent
PROVENANCE = json.loads((FOLDER / "provenance.json").read_text(encoding="utf-8"))

## 02 — Câu hỏi và giả thuyết
LR mặc định có cải thiện so với Dummy trên cùng người test không?
Kết quả có thể tăng hoặc giảm; không điều chỉnh thiết kế theo outer test.

## 03 — Dữ liệu
Bảng physiological68, giữ nguyên schema, checksum, nhãn và 15 subject theo cấu hình đã chạy. Không đưa metadata vào X.

In [2]:
feature_path = ROOT / CONFIG["features"]
if feature_path.is_file():
    assert hashlib.sha256(feature_path.read_bytes()).hexdigest() == CONFIG["feature_sha256"]
for name, expected in PROVENANCE["checksums"].items():
    assert hashlib.sha256((FOLDER / name).read_bytes()).hexdigest() == expected
print("PASS: hồ sơ kết quả và checksum.")

PASS: hồ sơ kết quả và checksum.


## 04 — Protocol
LOSO 15 người; binary và multiclass tách riêng. Imputer/scaler học trong train. Không tuning trong thí nghiệm này.

## 05 — Cấu hình
`config.yaml` là cấu hình đọc hồ sơ theo mã chung. `config.json` giữ nguyên manifest của lần chạy gốc; `config.yaml` dùng mã chung cho lần chạy mới.

In [3]:
display(CONFIG)

{'protocol': 'notebook_research_new_runs_v1', 'feature_version': 'physiological_68_v1', 'feature_sha256': '2f2fddd678f333a800f2869d2e78b407934b2c3492190ed70e28d8a9107fd8ba', 'source_paths': ['src/wesad/data.py', 'src/wesad/evaluation.py', 'src/wesad/models.py', 'src/wesad/utils.py'], 'source_hash_normalization': 'LF', 'expected_subjects': ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17'], 'models': ['lr'], 'tasks': ['multiclass', 'binary'], 'seed': 42, 'sensor_set': 'combined_physiological68', 'normalization': 'global_train', 'balance': False, 'nested': False, 'zero_division': 0, 'missing_train_class': 'error', 'missing_test_class': 'fixed_labels_for_macro_f1_observed_classes_for_balanced_accuracy', 'model_params': {'lr': {'random_state': 42}}, 'experiment_id': 'EXP-002', 'group_experiment_id': 'EXP-002', 'run_no': 'r02', 'run_scope': 'main', 'features': 'data/features/g2_physiological68_group15_20261107/all_subjects.csv', 'schema': 'data

## 06 — Thực thi
Notebook làm việc: `notebooks/experiments/EXP-002_working.ipynb`. Mặc định RUN_TRAIN=False; đọc kết quả đã lưu. Bật train sẽ dùng output mới, không ghi đè hồ sơ này.

## 07 — Kết quả
Mean và SD (ddof=1) tính theo 15 người; không dùng điểm gộp cửa sổ thay cho mean theo subject.

In [4]:
metrics = pd.read_csv(RESULTS_PATH)
assert len(metrics) == 30 and not metrics.duplicated(["task", "subject_id"]).any()
assert metrics.groupby("task").size().to_dict() == {"binary": 15, "multiclass": 15}
display(metrics)
summary = metrics.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
display(summary)

   subject_id        task model  n_test  macro_f1  balanced_accuracy  seconds
0         S10  multiclass    lr     426  0.290313           0.379056   0.5834
1         S11  multiclass    lr     417  0.593778           0.630667   0.8793
2         S13  multiclass    lr     417  0.582100           0.619231   0.6830
3         S14  multiclass    lr     410  0.469508           0.559466   0.7089
4         S15  multiclass    lr     418  0.707997           0.770603   0.5803
5         S16  multiclass    lr     415  0.363506           0.390855   0.7959
6         S17  multiclass    lr     426  0.637518           0.667492   0.8430
7          S2  multiclass    lr     395  0.466127           0.621005   0.6737
8          S3  multiclass    lr     401  0.661182           0.622287   0.5461
9          S4  multiclass    lr     404  0.701802           0.707692   0.6586
10         S5  multiclass    lr     414  0.796091           0.852285   0.6578
11         S6  multiclass    lr     412  0.786670           0.76

## 08 — Hình minh họa
Hình dựng từ kết quả/dự đoán đã lưu, dùng mã EXP chung. Không huấn luyện lại. Ảnh mã cũ trong thư mục chỉ dùng truy vết.

In [5]:
image_paths = sorted((FOLDER / "figures").glob(CONFIG["experiment_id"] + "_*.png"))
assert len(image_paths) == 5
for image_path in image_paths:
    display(Image(filename=str(image_path)))

<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 09 — Phân tích và đối chứng
Đối chứng chính: EXP-001. Chỉ so cùng task, subject và dữ liệu.

In [6]:
comparison_path = FOLDER / "comparison_by_subject.csv"
if comparison_path.is_file():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
    delta_columns = [c for c in comparison if c.endswith("_delta")]
    if delta_columns:
        display(comparison.groupby("task")[delta_columns].agg(["mean", "std"]))
for task, part in metrics.groupby("task"):
    display(part.nsmallest(3, "macro_f1")[["task", "subject_id", "macro_f1", "balanced_accuracy"]])

   subject_id        task  ...  macro_f1_delta  balanced_accuracy_delta
0         S10  multiclass  ...        0.059229                 0.045723
1         S11  multiclass  ...        0.358788                 0.297334
2         S13  multiclass  ...        0.347782                 0.285898
3         S14  multiclass  ...        0.237394                 0.226133
4         S15  multiclass  ...        0.474042                 0.437270
5         S16  multiclass  ...        0.128457                 0.057522
6         S17  multiclass  ...        0.406434                 0.334159
7          S2  multiclass  ...        0.228342                 0.287671
8          S3  multiclass  ...        0.426395                 0.288953
9          S4  multiclass  ...        0.465380                 0.374359
10         S5  multiclass  ...        0.557996                 0.518952
11         S6  multiclass  ...        0.550516                 0.430769
12         S7  multiclass  ...        0.408729                 0

## 10 — Kết luận và nguồn gốc
- binary: Macro-F1 0.810269 ± 0.193757; Balanced Accuracy 0.830362 ± 0.175180.
- multiclass: Macro-F1 0.589391 ± 0.148416; Balanced Accuracy 0.631471 ± 0.139367.

Kết quả biến động giữa người; không kết luận mô hình cuối chỉ từ một hướng. Chọn đặc trưng là nghiên cứu mở rộng sau khi đã quan sát test. Kết quả không phải kiểm định trên tập độc lập mới.

Nguồn cấu hình, checksum và commit được ghi tại `provenance.json`. So sánh train–test của EXP-001/002/011/016 ở `notebooks/04_evaluation/EXP_LR_train_test_comparison.ipynb`.